# CNN Pratiği: Kedi mi, Köpek mi? 🐱🐶

Bu notebook, **CNN ve YOLO** dersinde öğrenilen yöntemlerin kendi veri setime uygulamasıdır.
Derste hazır CIFAR-10 veri seti kullanılmıştı; burada **kendi görüntü dosyalarımla** (500 kedi +
500 köpek fotoğrafı) sıfırdan bir ikili sınıflandırma problemi çözüyorum.

> **Soru:** 1.000 fotoğraflık küçük bir veri setiyle kedi/köpek ayırt eden bir model eğitilebilir mi?
> Sıfırdan CNN mi daha iyi, yoksa transfer öğrenme mi?

**Dersle aynı iş akışı:** veri yükleme → görselleştirme → ön işleme → augmentasyon →
sıfırdan CNN → eğitim grafikleri → confusion matrix → transfer learning (MobileNetV2) →
karşılaştırma → YOLO ile nesne tespiti.

**Dersten farkı:** CIFAR-10 hazır ve 50.000 görüntülük bir veri setiydi (`keras.datasets` ile tek
satırda gelir, 32×32 piksel). Burada gerçek hayattaki gibi **kendi klasörlerimdeki değişken boyutlu
JPEG dosyalarıyla** çalışıyorum ve veri sadece 1.000 adet — bu, transfer öğrenmenin neden var
olduğunu gösteren mükemmel bir senaryo.

---
### ⚠️ Bu notebook Google Colab için yazılmıştır
Derin öğrenme modelleri GPU ister. Colab'da çalıştırmadan önce:
**Çalışma zamanı → Çalışma zamanı türünü değiştir → Donanım hızlandırıcı: GPU (T4)** seçin.

## Bölüm 0: Google Drive Bağlantısı ve Veri Hazırlığı

`archive__8_.zip` dosyasını Drive'a yükleyip aşağıdaki yolu ona göre düzenleyin.
Zip açıldığında `cats_set/` ve `dogs_set/` klasörleri oluşur — Keras bu yapıyı otomatik olarak
sınıf etiketine çevirir.

In [ ]:
# Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

# Zip dosyasının Drive'daki yolu (kendi klasörüne göre düzenle)
ZIP_YOLU = "/content/drive/MyDrive/SoftITO/archive__8_.zip"
VERI_KOK  = "/content/veri"   # zip buraya açılacak (Colab'ın hızlı yerel diski)

In [ ]:
import zipfile, os

os.makedirs(VERI_KOK, exist_ok=True)
with zipfile.ZipFile(ZIP_YOLU, "r") as z:
    z.extractall(VERI_KOK)

print("Açılan klasörler:", os.listdir(VERI_KOK))
for klasor in sorted(os.listdir(VERI_KOK)):
    yol = os.path.join(VERI_KOK, klasor)
    if os.path.isdir(yol):
        print(f"  {klasor}: {len(os.listdir(yol))} görüntü")

## Bölüm 1: Gerekli Kütüphaneleri İçe Aktarma

In [ ]:
import numpy as np                      # sayısal işlemler
import matplotlib.pyplot as plt          # görselleştirme
import tensorflow as tf                  # derin öğrenme çatısı
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("TensorFlow versiyonu:", tf.__version__)
print("GPU kullanılabilir mi:", len(tf.config.list_physical_devices('GPU')) > 0)

# Tekrarlanabilirlik için tohum
tf.random.set_seed(42)
np.random.seed(42)

## Bölüm 2: Veri Setini Yükleme ve Keşfetme

### 2.1: image_dataset_from_directory ile Yükleme

Derste CIFAR-10 hazır dizi (`array`) olarak geliyordu. Burada görüntüler diskte dosya halinde,
bu yüzden `image_dataset_from_directory` kullanıyoruz: klasör isimlerini otomatik olarak sınıf
etiketine çevirir, görüntüleri yeniden boyutlandırır ve verimli bir `tf.data` hattı kurar.

In [ ]:
IMG_BOYUT  = (160, 160)   # tüm görüntüler bu boyuta getirilecek
BATCH_SIZE = 32

# %80 eğitim / %20 doğrulama ayrımı (aynı seed ile tutarlı bölünme)
train_ds = keras.utils.image_dataset_from_directory(
    VERI_KOK,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_BOYUT,
    batch_size=BATCH_SIZE,
    label_mode="binary",      # 2 sınıf -> tek nöronlu çıkış (0/1)
)

val_ds = keras.utils.image_dataset_from_directory(
    VERI_KOK,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_BOYUT,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

sinif_isimleri = train_ds.class_names
print("\nSınıflar:", sinif_isimleri)   # ['cats_set', 'dogs_set'] -> 0 = kedi, 1 = köpek

### 2.2: Veri Setini Görselleştirme

Modeli eğitmeden önce veriye gözle bakmak şart — dersteki ilk kurallardan biri.

In [ ]:
plt.figure(figsize=(12, 8))
for goruntuler, etiketler in train_ds.take(1):
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(goruntuler[i].numpy().astype("uint8"))
        etiket = "Kedi" if etiketler[i].numpy()[0] == 0 else "Köpek"
        plt.title(etiket)
        plt.axis("off")
plt.suptitle("Eğitim Setinden Örnekler", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Sınıf dengesi kontrolü (dengesizlik varsa metrik seçimi değişirdi)
import collections
sayac = collections.Counter()
for _, etiketler in train_ds.unbatch():
    sayac[int(etiketler.numpy()[0])] += 1
print("Eğitim setindeki dağılım:", {sinif_isimleri[k]: v for k, v in sorted(sayac.items())})

### 2.3: Veri Ön İşleme ve Performans Ayarları

Piksel değerleri 0-255 aralığında; bunları 0-1'e ölçekliyoruz (dersteki normalizasyon adımı).
Ayrıca `cache` ve `prefetch` ile veri okuma darboğazını kaldırıyoruz — GPU beklemesin.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Normalizasyon katmanı (modelin ilk katmanı olarak eklenecek)
normalizasyon = layers.Rescaling(1.0 / 255)
print("Veri hattı hazır.")

### 2.4: Veri Augmentasyonu

Sadece 800 eğitim görüntümüz var — ezberleme (overfitting) riski yüksek. Augmentasyon, her epoch'ta
görüntüleri rastgele döndürüp çevirerek modele "yeni" örnekler gösterir. Dersteki
`ImageDataGenerator` yerine, modern Keras yaklaşımı olan **augmentasyon katmanlarını** kullanıyorum
(modelin içine gömülür, GPU'da çalışır).

In [ ]:
veri_augmentasyon = keras.Sequential([
    layers.RandomFlip("horizontal"),      # yatay çevirme (kedi sağa da bakabilir)
    layers.RandomRotation(0.15),          # hafif döndürme
    layers.RandomZoom(0.15),              # yakınlaştırma
    layers.RandomContrast(0.1),           # kontrast oynaması
], name="augmentasyon")

# Augmentasyonun etkisini tek bir görüntü üzerinde görelim
plt.figure(figsize=(10, 6))
for goruntuler, _ in train_ds.take(1):
    ilk = goruntuler[0]
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        artirilmis = veri_augmentasyon(tf.expand_dims(ilk, 0), training=True)
        plt.imshow(artirilmis[0].numpy().astype("uint8"))
        plt.axis("off")
plt.suptitle("Aynı Görüntünün 8 Farklı Augmentasyonu", fontsize=14)
plt.tight_layout()
plt.show()

## Bölüm 3: Sıfırdan CNN Modeli

### 3.1: Model Mimarisi

Dersteki mimarinin aynı mantığı: Conv2D (özellik çıkarma) → MaxPooling (boyut küçültme) blokları,
sonra Flatten + Dense (sınıflandırma). İkili sınıflandırma olduğu için çıkış katmanı **tek nöron +
sigmoid** (CIFAR-10'daki 10 nöron + softmax yerine).

In [ ]:
model_cnn = models.Sequential([
    layers.Input(shape=IMG_BOYUT + (3,)),
    veri_augmentasyon,                    # augmentasyon sadece eğitimde aktif olur
    normalizasyon,                        # 0-255 -> 0-1

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dropout(0.5),                  # nöronların yarısını rastgele kapat -> ezberlemeyi azaltır
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),  # 0 = kedi, 1 = köpek
], name="sifirdan_cnn")

model_cnn.summary()

### 3.2: Model Derleme ve Eğitim

İkili sınıflandırma → `binary_crossentropy` kayıp fonksiyonu.
`EarlyStopping` ile doğrulama kaybı iyileşmeyi bırakınca eğitim otomatik durur (dersteki callback).

In [ ]:
model_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

erken_durdurma = EarlyStopping(
    monitor="val_loss",
    patience=8,                  # 8 epoch boyunca iyileşme yoksa dur
    restore_best_weights=True,   # en iyi ağırlıklara geri dön
    verbose=1,
)

gecmis_cnn = model_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[erken_durdurma],
)

### 3.3: Eğitim Sonuçlarını Görselleştirme

Eğitim ve doğrulama eğrilerinin arasının açılması = ezberleme (overfitting) işareti.

In [ ]:
def egitim_grafigi(gecmis, baslik):
    acc      = gecmis.history["accuracy"]
    val_acc  = gecmis.history["val_accuracy"]
    loss     = gecmis.history["loss"]
    val_loss = gecmis.history["val_loss"]
    epochs   = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(epochs, acc, "o-", label="Eğitim", color="#3b6ea5")
    axes[0].plot(epochs, val_acc, "o-", label="Doğrulama", color="#d64545")
    axes[0].set_title("Doğruluk (Accuracy)"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(epochs, loss, "o-", label="Eğitim", color="#3b6ea5")
    axes[1].plot(epochs, val_loss, "o-", label="Doğrulama", color="#d64545")
    axes[1].set_title("Kayıp (Loss)"); axes[1].set_xlabel("Epoch"); axes[1].legend()

    plt.suptitle(baslik, fontsize=14)
    plt.tight_layout()
    plt.show()

egitim_grafigi(gecmis_cnn, "Sıfırdan CNN — Eğitim Geçmişi")

### 3.4: Değerlendirme, Confusion Matrix ve Sınıflandırma Raporu

In [ ]:
kayip_cnn, dogruluk_cnn = model_cnn.evaluate(val_ds, verbose=0)
print(f"Sıfırdan CNN — Doğrulama doğruluğu: {dogruluk_cnn:.4f} | Kayıp: {kayip_cnn:.4f}")

In [ ]:
# Gerçek etiketler ve tahminleri topla
y_gercek, y_olasilik = [], []
for goruntuler, etiketler in val_ds:
    y_gercek.extend(etiketler.numpy().ravel())
    y_olasilik.extend(model_cnn.predict(goruntuler, verbose=0).ravel())

y_gercek   = np.array(y_gercek).astype(int)
y_tahmin   = (np.array(y_olasilik) > 0.5).astype(int)   # 0.5 eşiği

cm = confusion_matrix(y_gercek, y_tahmin)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Kedi", "Köpek"], yticklabels=["Kedi", "Köpek"])
plt.xlabel("Tahmin"); plt.ylabel("Gerçek")
plt.title("Sıfırdan CNN — Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_gercek, y_tahmin, target_names=["Kedi", "Köpek"]))

## Bölüm 4: Transfer Learning — MobileNetV2

Sıfırdan CNN, 800 görüntüyle sıfırdan "kenar, doku, göz, kulak" öğrenmeye çalıştı — bu az veri için
zor bir görev. Transfer öğrenmede ise **1,4 milyon ImageNet görüntüsüyle eğitilmiş** MobileNetV2'nin
öğrendiği görsel özellikleri ödünç alıyoruz; sadece son sınıflandırma katmanını kendi problemimize
göre eğitiyoruz.

In [ ]:
# ImageNet ile önceden eğitilmiş MobileNetV2 (üst sınıflandırma katmanı olmadan)
temel_model = keras.applications.MobileNetV2(
    input_shape=IMG_BOYUT + (3,),
    include_top=False,        # kendi sınıflandırıcımızı ekleyeceğiz
    weights="imagenet",
)
temel_model.trainable = False   # önceden öğrenilen ağırlıklar donduruldu

print("Temel model katman sayısı:", len(temel_model.layers))
print("Dondurulmuş parametre sayısı:", f"{temel_model.count_params():,}")

In [ ]:
# MobileNetV2 kendi ön işlemesini ister: pikselleri [-1, 1] aralığına çeker
on_isleme = keras.applications.mobilenet_v2.preprocess_input

model_tl = models.Sequential([
    layers.Input(shape=IMG_BOYUT + (3,)),
    veri_augmentasyon,
    layers.Lambda(on_isleme),
    temel_model,
    layers.GlobalAveragePooling2D(),   # Flatten yerine: daha az parametre, daha az ezberleme
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
], name="transfer_learning")

model_tl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model_tl.summary()

In [ ]:
# Transfer learning çok daha az epoch ister
gecmis_tl = model_tl.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[EarlyStopping(monitor="val_loss", patience=4,
                             restore_best_weights=True, verbose=1)],
)

In [ ]:
egitim_grafigi(gecmis_tl, "Transfer Learning (MobileNetV2) — Eğitim Geçmişi")

kayip_tl, dogruluk_tl = model_tl.evaluate(val_ds, verbose=0)
print(f"Transfer Learning — Doğrulama doğruluğu: {dogruluk_tl:.4f} | Kayıp: {kayip_tl:.4f}")

In [ ]:
# Transfer learning modelinin confusion matrix'i
y_olasilik_tl = []
for goruntuler, _ in val_ds:
    y_olasilik_tl.extend(model_tl.predict(goruntuler, verbose=0).ravel())
y_tahmin_tl = (np.array(y_olasilik_tl) > 0.5).astype(int)

cm_tl = confusion_matrix(y_gercek, y_tahmin_tl)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_tl, annot=True, fmt="d", cmap="Greens",
            xticklabels=["Kedi", "Köpek"], yticklabels=["Kedi", "Köpek"])
plt.xlabel("Tahmin"); plt.ylabel("Gerçek")
plt.title("Transfer Learning — Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_gercek, y_tahmin_tl, target_names=["Kedi", "Köpek"]))

## Bölüm 5: İki Modelin Karşılaştırılması

In [ ]:
import pandas as pd

karsilastirma = pd.DataFrame({
    "Model": ["Sıfırdan CNN", "Transfer Learning (MobileNetV2)"],
    "Doğrulama Doğruluğu": [dogruluk_cnn, dogruluk_tl],
    "Doğrulama Kaybı": [kayip_cnn, kayip_tl],
    "Eğitilen Epoch": [len(gecmis_cnn.history["loss"]), len(gecmis_tl.history["loss"])],
    "Eğitilebilir Parametre": [f"{model_cnn.count_params():,}",
                               f"{sum(tf.size(w).numpy() for w in model_tl.trainable_weights):,}"],
})
display(karsilastirma.round(4))

# Görsel karşılaştırma
plt.figure(figsize=(7, 4.5))
plt.bar(["Sıfırdan CNN", "Transfer Learning"], [dogruluk_cnn, dogruluk_tl],
        color=["#3b6ea5", "#2e8b57"])
plt.ylabel("Doğrulama doğruluğu")
plt.ylim(0, 1)
for i, v in enumerate([dogruluk_cnn, dogruluk_tl]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=12)
plt.title("Model Karşılaştırması")
plt.tight_layout()
plt.show()

### Yanlış Sınıflandırılan Örnekler

Modelin nerede zorlandığını görmek, sayısal metrikten daha öğretici olabilir.

In [ ]:
# Doğrulama setindeki görüntüleri ve tahminleri tek bir listede topla
tum_goruntuler = []
for goruntuler, _ in val_ds:
    tum_goruntuler.extend(goruntuler.numpy())
tum_goruntuler = np.array(tum_goruntuler)

yanlislar = np.where(y_tahmin_tl != y_gercek)[0]
print(f"Transfer learning modelinin yanlış bildiği görüntü sayısı: {len(yanlislar)}")

if len(yanlislar) > 0:
    gosterilecek = yanlislar[:8]
    plt.figure(figsize=(13, 7))
    for i, idx in enumerate(gosterilecek):
        plt.subplot(2, 4, i + 1)
        plt.imshow(tum_goruntuler[idx].astype("uint8"))
        gercek  = "Kedi" if y_gercek[idx] == 0 else "Köpek"
        tahmin  = "Kedi" if y_tahmin_tl[idx] == 0 else "Köpek"
        plt.title(f"Gerçek: {gercek}\nTahmin: {tahmin} ({y_olasilik_tl[idx]:.2f})", fontsize=9)
        plt.axis("off")
    plt.suptitle("Yanlış Sınıflandırılan Örnekler", fontsize=14)
    plt.tight_layout()
    plt.show()

## Bölüm 6: YOLO ile Nesne Tespiti

CNN bir görüntüye tek etiket verir ("bu bir kedi"). YOLO ise **nesnenin görüntüde nerede olduğunu**
kutu çizerek gösterir ve birden fazla nesneyi aynı anda bulabilir. Dersteki YOLO bölümünü kendi
kedi/köpek fotoğraflarım üzerinde çalıştırıyorum.

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

# Önceden eğitilmiş YOLOv8 nano modeli (COCO veri seti ile eğitilmiş, 80 sınıf içerir)
yolo = YOLO("yolov8n.pt")

# COCO sınıfları arasında 'cat' (15) ve 'dog' (16) zaten var
print("COCO'daki hayvan sınıfları:",
      {k: v for k, v in yolo.names.items() if v in ["cat", "dog", "bird", "horse"]})

In [ ]:
import glob, random

# Kendi veri setimden rastgele 6 görüntü seç
tum_dosyalar = glob.glob(f"{VERI_KOK}/*/*.jpg")
random.seed(42)
secilenler = random.sample(tum_dosyalar, 6)

sonuclar = yolo(secilenler, verbose=False)

plt.figure(figsize=(15, 9))
for i, (yol, sonuc) in enumerate(zip(secilenler, sonuclar)):
    plt.subplot(2, 3, i + 1)
    plt.imshow(sonuc.plot()[:, :, ::-1])   # BGR -> RGB çevirimi
    tespitler = [f"{yolo.names[int(k.cls)]} {float(k.conf):.2f}" for k in sonuc.boxes]
    plt.title(", ".join(tespitler) if tespitler else "tespit yok", fontsize=10)
    plt.axis("off")
plt.suptitle("YOLOv8 ile Nesne Tespiti (kendi veri setimden)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# YOLO ne kadar başarılı? Klasör adını gerçek etiket kabul edip 60 görüntüde ölçelim
ornek_dosyalar = random.sample(tum_dosyalar, 60)
dogru, tespit_yok = 0, 0

for yol in ornek_dosyalar:
    gercek = "cat" if "cats_set" in yol else "dog"
    sonuc = yolo(yol, verbose=False)[0]
    etiketler = [yolo.names[int(k.cls)] for k in sonuc.boxes]
    hayvanlar = [e for e in etiketler if e in ("cat", "dog")]
    if not hayvanlar:
        tespit_yok += 1
    elif hayvanlar[0] == gercek:
        dogru += 1

print(f"60 görüntüde YOLO sonuçları:")
print(f"  Doğru tespit    : {dogru}")
print(f"  Hiç tespit yok  : {tespit_yok}")
print(f"  Yanlış tespit   : {60 - dogru - tespit_yok}")
print(f"\nNot: YOLO bu iş için ÖZEL eğitilmedi - COCO'dan gelen genel bilgiyle çalışıyor.")

## Bölüm 7: Modeli Kaydetme

In [ ]:
# En iyi modeli Drive'a kaydet (sonra tekrar eğitmeye gerek kalmaz)
KAYIT_YOLU = "/content/drive/MyDrive/SoftITO/kedi_kopek_transfer_model.keras"
model_tl.save(KAYIT_YOLU)
print("Model kaydedildi:", KAYIT_YOLU)

# Yeniden yükleme örneği:
# yuklenen = keras.models.load_model(KAYIT_YOLU)

## Özet — Bu Pratikte Öğrendiklerim

1. **Kendi görüntü dosyalarımla çalıştım** — CIFAR-10 gibi hazır veri yerine, klasör yapısından
   `image_dataset_from_directory` ile veri hattı kurdum (gerçek projelerdeki durum)
2. **İkili sınıflandırmaya uyarlama:** 10 nöron + softmax yerine tek nöron + sigmoid,
   `binary_crossentropy` kayıp fonksiyonu
3. **Augmentasyon katmanları** ile 800 görüntüden çok daha fazlasını ürettim; etkisini gözle gördüm
4. **Sıfırdan CNN** kurdum: Conv2D + MaxPooling blokları, Dropout ile ezberleme kontrolü
5. **EarlyStopping** ile gereksiz epoch'lar engellendi, en iyi ağırlıklar geri yüklendi
6. ⭐ **Transfer öğrenmenin gücü:** ImageNet'te eğitilmiş MobileNetV2'nin özelliklerini ödünç almak,
   az veriyle sıfırdan öğrenmeye göre çok daha etkili — az veri varsa transfer öğrenme ilk tercih
7. **Yanlış tahminleri görselleştirdim** — modelin nerede zorlandığını anlamak metrikten öğretici
8. **YOLO ile sınıflandırma-tespit farkını** gördüm: CNN "ne var?" der, YOLO "nerede?" diye de yanıtlar

**Sonraki adım fikirleri:** fine-tuning (MobileNetV2'nin üst katmanlarını da düşük öğrenme oranıyla
eğitmek), Grad-CAM ile modelin görüntünün neresine baktığını görselleştirmek, kendi verimle
YOLO eğitmek (etiketleme gerektirir).